# Object Detection and Depth Estimation

**Yolov8 for Object Detection**

**Models compared for Depth Estimation:**
1. StereoBM
2. StereoSGBM
3. MiDaS (monocular)
4. SIFT + Triangulation ----->
    a. SIFT_dense
    b. SIFT_sparse

In [1]:
import os, time, csv
import numpy as np
import cv2
import torch
from PIL import Image
from ultralytics import YOLO

KITTI_ROOT  = "/Users/anuragmahajan/Desktop/Analyze_Modules/data_scene_flow/training"
LEFT_DIR    = os.path.join(KITTI_ROOT, "image_2")
RIGHT_DIR   = os.path.join(KITTI_ROOT, "image_3")
GT_DISP_DIR = os.path.join(KITTI_ROOT, "disp_occ_0")
OUT_DIR     = "./depth_comparison_out"
QUAL_DIR    = os.path.join(OUT_DIR, "qualitative")

NUM_IMAGES      = None    
SAVE_QUAL_EVERY = 25

FOCAL_PX = 721.5377
BASELINE = 0.54

M_TO_MI = 0.000621371

DEPTH_MIN_M, DEPTH_MAX_M = 1.0, 80.0


YOLO_WEIGHTS = "yolov8n.pt"          
YOLO_CONF    = 0.35
VEHICLE_CLASSES = {2, 3, 5, 7}
VEHICLE_NAMES   = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}


def box_depth_stats(depth_map, valid_mask, x1, y1, x2, y2):
    H, W = depth_map.shape
    x1 = max(0, int(x1)); y1 = max(0, int(y1))
    x2 = min(W, int(x2)); y2 = min(H, int(y2))
    if x2 <= x1 or y2 <= y1:
        return None
    sub_d = depth_map[y1:y2, x1:x2]
    sub_v = valid_mask[y1:y2, x1:x2] & (sub_d >= DEPTH_MIN_M) & (sub_d <= DEPTH_MAX_M)
    if sub_v.sum() < 20:
        return None
    vals = sub_d[sub_v]
    return {
        "median_m":  float(np.median(vals)),
        "mean_m":    float(np.mean(vals)),
        "min_m":     float(np.min(vals)),
        "n_valid":   int(sub_v.sum()),
    }

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(QUAL_DIR, exist_ok=True)

# 1. StereoBM
bm = cv2.StereoBM_create(numDisparities=128, blockSize=9)
bm.setPreFilterCap(31); bm.setMinDisparity(0)
bm.setTextureThreshold(10); bm.setUniquenessRatio(15)
bm.setSpeckleWindowSize(100); bm.setSpeckleRange(32); bm.setDisp12MaxDiff(1)

def run_stereobm(left_bgr, right_bgr):
    gl = cv2.cvtColor(left_bgr,  cv2.COLOR_BGR2GRAY)
    gr = cv2.cvtColor(right_bgr, cv2.COLOR_BGR2GRAY)
    disp = bm.compute(gl, gr).astype(np.float32) / 16.0
    disp[disp <= 0] = 0.0
    return disp

# 2. StereoSGBM
SGBM_BS = 5
sgbm = cv2.StereoSGBM_create(
    minDisparity=0, numDisparities=128, blockSize=SGBM_BS,
    P1=8 * 3 * SGBM_BS * SGBM_BS, P2=32 * 3 * SGBM_BS * SGBM_BS,
    disp12MaxDiff=1, uniquenessRatio=10,
    speckleWindowSize=100, speckleRange=32,
    preFilterCap=63, mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY,
)

def run_stereosgbm(left_bgr, right_bgr):
    gl = cv2.cvtColor(left_bgr,  cv2.COLOR_BGR2GRAY)
    gr = cv2.cvtColor(right_bgr, cv2.COLOR_BGR2GRAY)
    disp = sgbm.compute(gl, gr).astype(np.float32) / 16.0
    disp[disp <= 0] = 0.0
    return disp

# 3. MiDaS (monocular)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[MiDaS] device: {DEVICE}")
midas = torch.hub.load("intel-isl/MiDaS", "DPT_Large").to(DEVICE).eval()
midas_tf = torch.hub.load("intel-isl/MiDaS", "transforms").dpt_transform

@torch.no_grad()
def run_midas(left_bgr):
    rgb = cv2.cvtColor(left_bgr, cv2.COLOR_BGR2RGB)
    inp = midas_tf(rgb).to(DEVICE)
    pred = midas(inp)
    pred = torch.nn.functional.interpolate(
        pred.unsqueeze(1), size=rgb.shape[:2],
        mode="bicubic", align_corners=False,
    ).squeeze().cpu().numpy()
    return pred  

def midas_align_to_gt(midas_pred, gt_depth, valid):
    inv_gt = 1.0 / gt_depth[valid]
    x = midas_pred[valid]
    A = np.stack([x, np.ones_like(x)], axis=1)
    s, t = np.linalg.lstsq(A, inv_gt, rcond=None)[0]
    inv_pred = np.clip(s * midas_pred + t, 1.0 / DEPTH_MAX_M, 1.0 / DEPTH_MIN_M)
    return 1.0 / inv_pred

# 4. SIFT + Triangulation
sift  = cv2.SIFT_create(nfeatures=4000)
flann = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50))

def run_sift_triangulation(left_bgr, right_bgr, ratio=0.75, max_y=2.0):
    gl = cv2.cvtColor(left_bgr,  cv2.COLOR_BGR2GRAY)
    gr = cv2.cvtColor(right_bgr, cv2.COLOR_BGR2GRAY)
    kpL, dL = sift.detectAndCompute(gl, None)
    kpR, dR = sift.detectAndCompute(gr, None)
    H, W = gl.shape
    sd = np.zeros((H, W), np.float32)
    sm = np.zeros((H, W), bool)
    if dL is None or dR is None or len(kpL) < 2 or len(kpR) < 2:
        return sd, sm
    for pair in flann.knnMatch(dL, dR, k=2):
        if len(pair) < 2: continue
        m, n = pair
        if m.distance >= ratio * n.distance: continue
        xL, yL = kpL[m.queryIdx].pt
        xR, yR = kpR[m.trainIdx].pt
        if abs(yL - yR) > max_y: continue
        d = xL - xR
        if d <= 0: continue
        xi, yi = int(round(xL)), int(round(yL))
        if 0 <= xi < W and 0 <= yi < H:
            sd[yi, xi] = d
            sm[yi, xi] = True
    return sd, sm

def densify_nearest(sparse_disp, sparse_mask):
    if not sparse_mask.any():
        return sparse_disp.copy()
    inv = (~sparse_mask).astype(np.uint8)
    _, labels = cv2.distanceTransformWithLabels(
        inv, cv2.DIST_L2, 3, labelType=cv2.DIST_LABEL_PIXEL)
    ys, xs = np.where(sparse_mask)
    seed_labels = labels[ys, xs]
    lut = np.zeros(labels.max() + 1, dtype=np.float32)
    lut[seed_labels] = sparse_disp[ys, xs]
    return lut[labels]

#YOLOv8 (vehicle detection on the left image)
print(f"[YOLOv8] loading {YOLO_WEIGHTS} on {DEVICE}")
yolo = YOLO(YOLO_WEIGHTS)

def run_yolo_vehicles(left_bgr):
    """Return list of dicts: {cls, name, conf, x1,y1,x2,y2}."""
    res = yolo.predict(left_bgr, conf=YOLO_CONF, device=DEVICE, verbose=False)[0]
    out = []
    if res.boxes is None or len(res.boxes) == 0:
        return out
    xyxy = res.boxes.xyxy.cpu().numpy()
    cls  = res.boxes.cls.cpu().numpy().astype(int)
    conf = res.boxes.conf.cpu().numpy()
    for (x1, y1, x2, y2), c, p in zip(xyxy, cls, conf):
        if c not in VEHICLE_CLASSES:
            continue
        out.append({
            "cls": int(c), "name": VEHICLE_NAMES.get(int(c), str(c)),
            "conf": float(p),
            "x1": float(x1), "y1": float(y1), "x2": float(x2), "y2": float(y2),
        })
    return out

def load_gt_disp(path):
    img = np.array(Image.open(path)).astype(np.float32)
    return img / 256.0, img > 0

def disp_to_depth(disp):
    z = np.zeros_like(disp, np.float32)
    nz = disp > 0
    z[nz] = (FOCAL_PX * BASELINE) / disp[nz]
    return z

def compute_metrics(pred_disp, pred_mask, gt_disp, gt_mask):
    gt_depth_m   = disp_to_depth(gt_disp)
    pred_depth_m = disp_to_depth(pred_disp)

    mask = (gt_mask & pred_mask & (pred_disp > 0)
            & (gt_depth_m   >= DEPTH_MIN_M) & (gt_depth_m   <= DEPTH_MAX_M)
            & (pred_depth_m >= DEPTH_MIN_M) & (pred_depth_m <= DEPTH_MAX_M))
    if mask.sum() < 100:
        return None

    z_gt   = gt_depth_m[mask]   * M_TO_MI
    z_pred = pred_depth_m[mask] * M_TO_MI

    abs_rel = float(np.mean(np.abs(z_pred - z_gt) / z_gt))      
    rmse_mi = float(np.sqrt(np.mean((z_pred - z_gt) ** 2)))     
    ratio   = np.maximum(z_pred / z_gt, z_gt / z_pred)
    delta1  = float((ratio < 1.25).mean())                      
    density = float(mask.sum() / max(gt_mask.sum(), 1))

    return {"AbsRel": abs_rel, "RMSE_mi": rmse_mi,
            "delta1": delta1, "density": density}

def depth_to_color(depth_m):
    z = np.clip(depth_m, DEPTH_MIN_M, DEPTH_MAX_M)
    z = ((z - DEPTH_MIN_M) / (DEPTH_MAX_M - DEPTH_MIN_M) * 255).astype(np.uint8)
    return cv2.applyColorMap(z, cv2.COLORMAP_PLASMA)

left_files = sorted(f for f in os.listdir(LEFT_DIR) if f.endswith("_10.png"))
if NUM_IMAGES is not None:
    left_files = left_files[:NUM_IMAGES]

per_image_rows   = []   
per_vehicle_rows = []  

for idx, fname in enumerate(left_files):
    lp = os.path.join(LEFT_DIR,  fname)
    rp = os.path.join(RIGHT_DIR, fname)
    gp = os.path.join(GT_DISP_DIR, fname)
    if not (os.path.exists(rp) and os.path.exists(gp)):
        continue

    left  = cv2.imread(lp)
    right = cv2.imread(rp)
    gt_disp, gt_mask = load_gt_disp(gp)

    # detect vehicles on the left image
    t0 = time.time()
    vehicles = run_yolo_vehicles(left)
    yolo_t = time.time() - t0

    t0 = time.time(); bm_disp = run_stereobm(left, right); bm_t = time.time() - t0
    bm_m = compute_metrics(bm_disp, bm_disp > 0, gt_disp, gt_mask)

    t0 = time.time(); sgbm_disp = run_stereosgbm(left, right); sgbm_t = time.time() - t0
    sgbm_m = compute_metrics(sgbm_disp, sgbm_disp > 0, gt_disp, gt_mask)

    t0 = time.time()
    midas_raw = run_midas(left)
    gt_depth_full = disp_to_depth(gt_disp)
    align_mask = (gt_mask
                  & (gt_depth_full >= DEPTH_MIN_M)
                  & (gt_depth_full <= DEPTH_MAX_M))
    midas_depth = midas_align_to_gt(midas_raw, gt_depth_full, align_mask)
    midas_disp  = (FOCAL_PX * BASELINE) / np.clip(midas_depth, 1e-3, None)
    midas_t = time.time() - t0
    midas_m = compute_metrics(midas_disp, np.ones_like(midas_disp, bool),
                              gt_disp, gt_mask)

    t0 = time.time()
    sift_sparse, sift_mask = run_sift_triangulation(left, right)
    sift_dense = densify_nearest(sift_sparse, sift_mask)
    sift_t = time.time() - t0
    sift_dense_m  = compute_metrics(sift_dense, sift_dense > 0, gt_disp, gt_mask)
    sift_sparse_m = compute_metrics(sift_sparse, sift_mask,    gt_disp, gt_mask)

    for name, m, t in [
        ("StereoBM",    bm_m,         bm_t),
        ("StereoSGBM",  sgbm_m,       sgbm_t),
        ("MiDaS",       midas_m,      midas_t),
        ("SIFT_dense",  sift_dense_m, sift_t),
        ("SIFT_sparse", sift_sparse_m, sift_t),
    ]:
        if m is None: continue
        row = {"image": fname, "method": name, "time_s": t}
        row.update(m)
        per_image_rows.append(row)

    bm_depth    = disp_to_depth(bm_disp)
    sgbm_depth  = disp_to_depth(sgbm_disp)
    sift_depth  = disp_to_depth(sift_dense)
    gt_depth_m  = disp_to_depth(gt_disp)

    method_depths = [
        ("StereoBM",   bm_depth,    bm_disp   > 0),
        ("StereoSGBM", sgbm_depth,  sgbm_disp > 0),
        ("MiDaS",      midas_depth, np.ones_like(midas_depth, bool)),
        ("SIFT_dense", sift_depth,  sift_dense > 0),
        ("GT",         gt_depth_m,  gt_mask),
    ]

    for vi, v in enumerate(vehicles):
        for mname, dmap, vmask in method_depths:
            stats = box_depth_stats(dmap, vmask, v["x1"], v["y1"], v["x2"], v["y2"])
            if stats is None:
                continue
            per_vehicle_rows.append({
                "image":    fname,
                "veh_id":   vi,
                "class":    v["name"],
                "conf":     round(v["conf"], 3),
                "x1": int(v["x1"]), "y1": int(v["y1"]),
                "x2": int(v["x2"]), "y2": int(v["y2"]),
                "method":   mname,
                "median_m": round(stats["median_m"], 2),
                "mean_m":   round(stats["mean_m"],   2),
                "min_m":    round(stats["min_m"],    2),
                "median_mi":round(stats["median_m"] * M_TO_MI, 5),
                "n_valid":  stats["n_valid"],
            })

    if idx % SAVE_QUAL_EVERY == 0:
        gt_z = disp_to_depth(gt_disp); gt_z[~gt_mask] = 0

        left_annot = left.copy()
        for v in vehicles:
            x1, y1, x2, y2 = map(int, (v["x1"], v["y1"], v["x2"], v["y2"]))
            cv2.rectangle(left_annot, (x1, y1), (x2, y2), (0, 255, 0), 2)
            s = box_depth_stats(sgbm_depth, sgbm_disp > 0, x1, y1, x2, y2)
            label = f"{v['name']}"
            if s is not None:
                label += f" {s['median_m']:.1f}m"
            cv2.putText(left_annot, label, (x1, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        panel = np.vstack([
            np.hstack([left_annot,                       depth_to_color(gt_z)]),
            np.hstack([depth_to_color(bm_depth),
                       depth_to_color(sgbm_depth)]),
            np.hstack([depth_to_color(midas_depth),
                       depth_to_color(sift_depth)]),
        ])
        cv2.imwrite(os.path.join(QUAL_DIR, f"{idx:03d}_{fname}"), panel)

    print(f"[{idx+1}/{len(left_files)}] {fname}  "
          f"vehicles={len(vehicles)}  "
          f"BM AbsRel={bm_m['AbsRel']:.3f}  "
          f"SGBM AbsRel={sgbm_m['AbsRel']:.3f}  "
          f"MiDaS AbsRel={midas_m['AbsRel']:.3f}  "
          f"SIFT(d) AbsRel={sift_dense_m['AbsRel']:.3f}  "
          f"yolo={yolo_t:.2f}s")

fields = ["image", "method", "time_s", "AbsRel", "RMSE_mi", "delta1", "density"]
with open(os.path.join(OUT_DIR, "results.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
    for r in per_image_rows: w.writerow(r)

groups = {}
for r in per_image_rows:
    groups.setdefault(r["method"], []).append(r)

with open(os.path.join(OUT_DIR, "summary.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["method", "n_images",
                "AbsRel_down", "RMSE_mi_down", "delta1_up",
                "density", "time_s"])
    for method, rows in groups.items():
        n = len(rows)
        avg = lambda k: float(np.mean([r[k] for r in rows]))
        w.writerow([method, n,
                    f"{avg('AbsRel'):.4f}",
                    f"{avg('RMSE_mi'):.6f}",   
                    f"{avg('delta1'):.4f}",
                    f"{avg('density'):.3f}",
                    f"{avg('time_s'):.3f}"])

veh_fields = ["image", "veh_id", "class", "conf",
              "x1", "y1", "x2", "y2",
              "method", "median_m", "mean_m", "min_m", "median_mi", "n_valid"]
with open(os.path.join(OUT_DIR, "vehicles.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=veh_fields); w.writeheader()
    for r in per_vehicle_rows: w.writerow(r)

gt_lookup = {}
for r in per_vehicle_rows:
    if r["method"] == "GT":
        gt_lookup[(r["image"], r["veh_id"])] = r["median_m"]

veh_err_rows = []
for r in per_vehicle_rows:
    if r["method"] == "GT": continue
    key = (r["image"], r["veh_id"])
    if key not in gt_lookup: continue
    gt_z = gt_lookup[key]
    if gt_z <= 0: continue
    err = r["median_m"] - gt_z
    veh_err_rows.append({
        "method": r["method"],
        "abs_err_m": abs(err),
        "rel_err": abs(err) / gt_z,
    })

veh_groups = {}
for r in veh_err_rows:
    veh_groups.setdefault(r["method"], []).append(r)

with open(os.path.join(OUT_DIR, "vehicles_summary.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["method", "n_vehicles", "MAE_m", "MeanRelErr", "MedianRelErr"])
    for method, rows in veh_groups.items():
        n = len(rows)
        mae   = float(np.mean([r["abs_err_m"] for r in rows]))
        mre   = float(np.mean([r["rel_err"]   for r in rows]))
        medre = float(np.median([r["rel_err"] for r in rows]))
        w.writerow([method, n, f"{mae:.3f}", f"{mre:.4f}", f"{medre:.4f}"])

print("\nDone.")
print(f"Per-image    : {os.path.join(OUT_DIR, 'results.csv')}")
print(f"Summary      : {os.path.join(OUT_DIR, 'summary.csv')}")
print(f"Per-vehicle  : {os.path.join(OUT_DIR, 'vehicles.csv')}")
print(f"Vehicle summ.: {os.path.join(OUT_DIR, 'vehicles_summary.csv')}")
print(f"Visuals      : {QUAL_DIR}")

[MiDaS] device: cpu


Using cache found in /Users/anuragmahajan/.cache/torch/hub/intel-isl_MiDaS_master
/Users/anuragmahajan/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using cache found in /Users/anuragmahajan/.cache/torch/hub/intel-isl_MiDaS_master


[YOLOv8] loading yolov8n.pt on cpu
[1/200] 000000_10.png  vehicles=9  BM AbsRel=0.035  SGBM AbsRel=0.036  MiDaS AbsRel=0.072  SIFT(d) AbsRel=0.227  yolo=0.10s
[2/200] 000001_10.png  vehicles=3  BM AbsRel=0.032  SGBM AbsRel=0.044  MiDaS AbsRel=0.119  SIFT(d) AbsRel=0.208  yolo=0.03s
[3/200] 000002_10.png  vehicles=1  BM AbsRel=0.046  SGBM AbsRel=0.055  MiDaS AbsRel=0.074  SIFT(d) AbsRel=0.244  yolo=0.03s
[4/200] 000003_10.png  vehicles=3  BM AbsRel=0.069  SGBM AbsRel=0.092  MiDaS AbsRel=0.150  SIFT(d) AbsRel=0.234  yolo=0.03s
[5/200] 000004_10.png  vehicles=11  BM AbsRel=0.041  SGBM AbsRel=0.060  MiDaS AbsRel=0.186  SIFT(d) AbsRel=0.203  yolo=0.03s
[6/200] 000005_10.png  vehicles=13  BM AbsRel=0.032  SGBM AbsRel=0.044  MiDaS AbsRel=0.177  SIFT(d) AbsRel=0.180  yolo=0.05s
[7/200] 000006_10.png  vehicles=12  BM AbsRel=0.042  SGBM AbsRel=0.062  MiDaS AbsRel=0.152  SIFT(d) AbsRel=0.231  yolo=0.03s
[8/200] 000007_10.png  vehicles=2  BM AbsRel=0.048  SGBM AbsRel=0.049  MiDaS AbsRel=0.076  SIF